# Residual geometry and transfer: S1 vs voice (0.5B)

Two different stenographies, same residual-space test:

- **S1 (positional / semantic):** the final answer follows the first sentence.
- **Voice (syntactic):** active → 1, passive → 0, meaning held fixed.

This notebook does **not** reuse the paired-val file to fit directions.

1. Fit each residual arrow on **train** pairs (discovery).
2. Keep **val** only for geometry hold-out checks and causal projection.
3. Plot the two arrows (cosine, PCA of per-pair differences, SVD).
4. Project out `d_S1`, `d_voice`, the **shared** SVD axis, and each **private** leftover — on both adapters, at layers 18 and 19.

A shared axis is interesting only if it moves **both** rules without collapsing every pair to the same label.

Colab: **Runtime → GPU (T4)**.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"
!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!pip install -q peft "transformers<4.50" accelerate matplotlib
!pip uninstall -y torchao >/dev/null 2>&1

In [ ]:
from pathlib import Path
from google.colab import files
from transformers import AutoTokenizer
import hashlib, json

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
S1_ADAPTER = Path("checkpoints/qwen05b-cot-sft-v2")
VOICE_ADAPTER = Path("checkpoints/qwen05b-cot-sft-voice-paired")
LAYERS_TO_INTERVENE = (18, 19)
DISCOVERY_N = 80
OUT = Path("/content/residual_s1_voice_transfer_05b")
OUT.mkdir(parents=True, exist_ok=True)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def ensure_adapter(dest: Path, label: str):
    dest.mkdir(parents=True, exist_ok=True)
    weights = dest / "adapter_model.safetensors"
    config = dest / "adapter_config.json"
    if weights.is_file() and config.is_file():
        print(f"Using existing {label} adapter at {dest}")
    else:
        print(f"Upload {label} adapter_config.json and adapter_model.safetensors")
        uploaded = files.upload()
        for name, data in uploaded.items():
            if Path(name).name in {"adapter_config.json", "adapter_model.safetensors", "training_log.json"}:
                (dest / Path(name).name).write_bytes(data)
    assert config.is_file() and weights.is_file(), dest
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(dest)
    return sha256(weights)


S1_SHA = ensure_adapter(S1_ADAPTER, "S1 (qwen05b-cot-sft-v2)")
VOICE_SHA = ensure_adapter(VOICE_ADAPTER, "voice (qwen05b-cot-sft-voice-paired / minimal)")
assert S1_SHA != VOICE_SHA, "The two adapters are identical"
print("S1    ", S1_SHA[:12])
print("voice ", VOICE_SHA[:12])

In [ ]:
import sys
from evaluation.evaluate_ethics_morality import build_prompt
from intervention.cot_utils import split_sentences
from intervention.make_minimal_negations import swap_verdict

CANONICAL_S1 = {
    0: "The described action is perfectly fine.",
    1: "The described action is morally wrong.",
}


def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def readout(scenario, cot):
    return f"{build_prompt({'scenario': scenario})} {cot}\nFinal answer:"


def voice_pairs(path):
    groups = {}
    for row in load_jsonl(path):
        groups.setdefault(int(row["pair_index"]), {})[row["voice"]] = row
    pairs = []
    for pair_index, pair in sorted(groups.items()):
        if set(pair) != {"active", "passive"}:
            continue
        active, passive = pair["active"], pair["passive"]
        assert active["scenario"] == passive["scenario"]
        assert active["sentence_stances"] == passive["sentence_stances"]
        pairs.append({
            "pair_index": pair_index,
            "scenario": active["scenario"],
            "pos_text": readout(active["scenario"], active["chain_of_thought"]),
            "neg_text": readout(passive["scenario"], passive["chain_of_thought"]),
            "pos_label": 1,
            "neg_label": 0,
            "flip": "voice",
        })
    return pairs


def make_s1_pair(row):
    sentences = list(row.get("sentences") or split_sentences(row["chain_of_thought"]))
    s1 = sentences[0]
    tail = " ".join(sentences[1:])
    stance = int(row.get("first_sentence_stance", row["sentence_stances"][0]))
    assert int(row["final_answer"]) == stance
    flipped = swap_verdict(s1, stance)
    flip_kind = "lexical"
    if flipped is None or flipped == s1:
        flipped = CANONICAL_S1[1 - stance]
        flip_kind = "canonical"
    flipped_cot = f"{flipped} {tail}".strip() if tail else flipped
    cot_by_stance = {stance: row["chain_of_thought"], 1 - stance: flipped_cot}
    return {
        "pair_index": int(row["index"]),
        "scenario": row["scenario"],
        "pos_text": readout(row["scenario"], cot_by_stance[1]),
        "neg_text": readout(row["scenario"], cot_by_stance[0]),
        "pos_label": 1,
        "neg_label": 0,
        "flip": flip_kind,
    }


def s1_pairs(path):
    lexical, fallback = [], []
    for row in load_jsonl(path):
        pair = make_s1_pair(row)
        (lexical if pair["flip"] == "lexical" else fallback).append(pair)
    return lexical + fallback


voice_discovery = voice_pairs(Path("data/training_data/synthetic_ethics_voice_paired_train.jsonl"))[:DISCOVERY_N]
voice_eval = voice_pairs(Path("data/validation_data/synthetic_ethics_voice_paired_val.jsonl"))
s1_train_all = s1_pairs(Path("data/training_data/synthetic_ethics_cot_training_v2.jsonl"))
s1_discovery = s1_train_all[:DISCOVERY_N]
s1_eval = s1_pairs(Path("data/validation_data/synthetic_ethics_cot_val_v2.jsonl"))

assert len(voice_discovery) == DISCOVERY_N, len(voice_discovery)
assert len(s1_discovery) == DISCOVERY_N, len(s1_discovery)
assert len(voice_eval) == 100, len(voice_eval)
assert len(s1_eval) == 50, len(s1_eval)

TASKS = {
    "s1": {"discovery": s1_discovery, "eval": s1_eval, "adapter": S1_ADAPTER},
    "voice": {"discovery": voice_discovery, "eval": voice_eval, "adapter": VOICE_ADAPTER},
}
print("voice discovery/eval", len(voice_discovery), len(voice_eval))
print("S1    discovery/eval", len(s1_discovery), len(s1_eval))
print("S1 discovery flips", {k: sum(p['flip']==k for p in s1_discovery) for k in ("lexical", "canonical")})
print("S1 eval flips      ", {k: sum(p['flip']==k for p in s1_eval) for k in ("lexical", "canonical")})

In [ ]:
import gc
import torch.nn.functional as F
from sparse_autoencoders.run_sae import load_model, transformer_layers

DEVICE = torch.device("cuda")
BATCH_SIZE = 8


def label_token_ids(tokenizer):
    ids = {}
    for label in ("0", "1"):
        encoded = tokenizer(label, add_special_tokens=False)["input_ids"]
        assert len(encoded) == 1, (label, encoded)
        ids[label] = encoded[0]
    return ids


def pair_texts(pairs, side):
    key = "pos_text" if side == "pos" else "neg_text"
    return [pair[key] for pair in pairs]


@torch.no_grad()
def collect_all_layers(model, tokenizer, texts):
    tokenizer.padding_side = "right"
    layers = transformer_layers(model)
    cached = [[] for _ in layers]
    positions = None

    def make_hook(layer_index):
        def hook(_module, _inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            index = torch.arange(hidden.shape[0], device=hidden.device)
            cached[layer_index].append(hidden[index, positions].detach().float().cpu())
        return hook

    handles = [layer.register_forward_hook(make_hook(i)) for i, layer in enumerate(layers)]
    margins = []
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        for handle in handles:
            handle.remove()
    activations = torch.stack([torch.cat(rows) for rows in cached], dim=1)
    return activations, torch.cat(margins)


def collect_task(model, tokenizer, pairs):
    pos_h, pos_m = collect_all_layers(model, tokenizer, pair_texts(pairs, "pos"))
    neg_h, neg_m = collect_all_layers(model, tokenizer, pair_texts(pairs, "neg"))
    return {"pos_h": pos_h, "neg_h": neg_h, "pos_margin": pos_m, "neg_margin": neg_m}


def unload(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


print("Collecting unadapted base (voice texts, then S1 texts)...")
base_tok, base_model = load_model(BASE_MODEL, DEVICE)
base = {
    "voice": {
        "discovery": collect_task(base_model, base_tok, voice_discovery),
        "eval": collect_task(base_model, base_tok, voice_eval),
    },
    "s1": {
        "discovery": collect_task(base_model, base_tok, s1_discovery),
        "eval": collect_task(base_model, base_tok, s1_eval),
    },
}
unload(base_model, base_tok)

print("Collecting S1 adapter...")
s1_tok, s1_model = load_model(str(S1_ADAPTER), DEVICE)
adapter_acts = {
    "s1": {
        "discovery": collect_task(s1_model, s1_tok, s1_discovery),
        "eval": collect_task(s1_model, s1_tok, s1_eval),
    }
}
unload(s1_model, s1_tok)

print("Collecting voice adapter...")
voice_tok, voice_model = load_model(str(VOICE_ADAPTER), DEVICE)
adapter_acts["voice"] = {
    "discovery": collect_task(voice_model, voice_tok, voice_discovery),
    "eval": collect_task(voice_model, voice_tok, voice_eval),
}
unload(voice_model, voice_tok)

n_layers = adapter_acts["voice"]["discovery"]["pos_h"].shape[1]
hidden = adapter_acts["voice"]["discovery"]["pos_h"].shape[-1]
print(f"layers={n_layers} hidden={hidden}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def gap(bundle):
    return float(bundle["pos_margin"].mean() - bundle["neg_margin"].mean())


def pair_dod(adapter_bundle, base_bundle, layer):
    return (
        (adapter_bundle["pos_h"][:, layer] - adapter_bundle["neg_h"][:, layer])
        - (base_bundle["pos_h"][:, layer] - base_bundle["neg_h"][:, layer])
    )


def mean_direction(dod):
    return F.normalize(dod.mean(0), dim=0)


def adapter_center(adapter_bundle, layer):
    return torch.cat([adapter_bundle["pos_h"][:, layer], adapter_bundle["neg_h"][:, layer]]).mean(0)


def rule_stats(bundle):
    pos_one = bundle["pos_margin"] > 0
    neg_one = bundle["neg_margin"] > 0
    return {
        "gap": gap(bundle),
        "rule_accuracy": float(torch.cat([pos_one, ~neg_one]).float().mean()),
        "pos_predicts_1": float(pos_one.float().mean()),
        "neg_predicts_1": float(neg_one.float().mean()),
        "pair_agreement": float((pos_one == neg_one).float().mean()),
        "both_0": int((~pos_one & ~neg_one).sum()),
        "both_1": int((pos_one & neg_one).sum()),
        "pos1_neg0": int((pos_one & ~neg_one).sum()),
        "pos0_neg1": int((~pos_one & neg_one).sum()),
    }


layer_geometry = []
directions = {}
for layer in range(n_layers):
    d_s1_pairs = pair_dod(adapter_acts["s1"]["discovery"], base["s1"]["discovery"], layer)
    d_voice_pairs = pair_dod(adapter_acts["voice"]["discovery"], base["voice"]["discovery"], layer)
    d_s1 = mean_direction(d_s1_pairs)
    d_voice = mean_direction(d_voice_pairs)
    stacked = torch.stack([d_s1, d_voice], dim=0)
    _, singular, vh = torch.linalg.svd(stacked, full_matrices=False)
    shared = F.normalize(vh[0], dim=0)
    if (d_s1 * shared).sum() < 0:
        shared = -shared
    private_s1 = F.normalize(d_s1 - (d_s1 * shared).sum() * shared, dim=0)
    private_voice = F.normalize(d_voice - (d_voice * shared).sum() * shared, dim=0)
    directions[layer] = {
        "s1": d_s1,
        "voice": d_voice,
        "shared": shared,
        "private_s1": private_s1,
        "private_voice": private_voice,
        "center_s1": adapter_center(adapter_acts["s1"]["discovery"], layer),
        "center_voice": adapter_center(adapter_acts["voice"]["discovery"], layer),
        "pair_dod_s1": d_s1_pairs,
        "pair_dod_voice": d_voice_pairs,
    }
    layer_geometry.append({
        "layer": layer,
        "cosine_s1_voice": float((d_s1 * d_voice).sum()),
        "shared_sv0_fraction": float(singular[0] / singular.sum()),
        "s1_on_shared": float((d_s1 * shared).sum()),
        "voice_on_shared": float((d_voice * shared).sum()),
        "s1_adapter_eval_gap": gap(adapter_acts["s1"]["eval"]) if layer == 0 else None,
        "voice_adapter_eval_gap": gap(adapter_acts["voice"]["eval"]) if layer == 0 else None,
    })

(OUT / "layer_geometry.json").write_text(json.dumps([
    {
        "layer": row["layer"],
        "cosine_s1_voice": row["cosine_s1_voice"],
        "shared_sv0_fraction": row["shared_sv0_fraction"],
        "s1_on_shared": row["s1_on_shared"],
        "voice_on_shared": row["voice_on_shared"],
    }
    for row in layer_geometry
], indent=2))

baseline_eval = {
    "s1_base": rule_stats(base["s1"]["eval"]),
    "s1_adapter": rule_stats(adapter_acts["s1"]["eval"]),
    "voice_base": rule_stats(base["voice"]["eval"]),
    "voice_adapter": rule_stats(adapter_acts["voice"]["eval"]),
}
(OUT / "baseline_eval_stats.json").write_text(json.dumps(baseline_eval, indent=2))
print(json.dumps(baseline_eval, indent=2))
print("cosine L18", layer_geometry[18]["cosine_s1_voice"])
print("cosine L19", layer_geometry[19]["cosine_s1_voice"])

In [ ]:
cosines = [row["cosine_s1_voice"] for row in layer_geometry]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
axes[0].plot(range(n_layers), cosines, color="tab:blue")
for layer in LAYERS_TO_INTERVENE:
    axes[0].axvline(layer, color="tab:red", linestyle="--", linewidth=1)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(title="Cosine between mean residual arrows", xlabel="Layer", ylabel="cos(d_S1, d_voice)")

plot_layer = 19 if abs(layer_geometry[19]["cosine_s1_voice"]) >= abs(layer_geometry[18]["cosine_s1_voice"]) else 18
dod = torch.cat([
    directions[plot_layer]["pair_dod_s1"],
    directions[plot_layer]["pair_dod_voice"],
], dim=0).numpy()
dod = dod - dod.mean(0, keepdims=True)
_, _, vt = np.linalg.svd(dod, full_matrices=False)
xy = dod @ vt[:2].T
n_s1 = directions[plot_layer]["pair_dod_s1"].shape[0]
axes[1].scatter(xy[:n_s1, 0], xy[:n_s1, 1], s=18, alpha=0.75, label="S1 pairs", color="tab:orange")
axes[1].scatter(xy[n_s1:, 0], xy[n_s1:, 1], s=18, alpha=0.75, label="Voice pairs", color="tab:green")
axes[1].set(title=f"Per-pair DoD PCA at layer {plot_layer}", xlabel="PC1", ylabel="PC2")
axes[1].legend(frameon=False)
fig.suptitle("Residual geometry (arrows fit on train / discovery)")
fig.savefig(OUT / "residual_geometry.png", dpi=300, bbox_inches="tight")
plt.show()
print("PCA layer", plot_layer)

In [ ]:
@torch.no_grad()
def evaluate_projection(model, tokenizer, texts, *, layer, direction, center):
    tokenizer.padding_side = "right"
    positions = None
    direction = direction.to(DEVICE)
    center = center.to(DEVICE)
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        index = torch.arange(hidden.shape[0], device=hidden.device)
        target = hidden[index, positions].float()
        coefficient = ((target - center) * direction).sum(-1, keepdim=True)
        patched = hidden.clone()
        patched[index, positions] = (target - coefficient * direction).to(hidden.dtype)
        return (patched,) + output[1:] if isinstance(output, tuple) else patched

    handle = transformer_layers(model)[layer].register_forward_hook(hook)
    margins = []
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        handle.remove()
    return torch.cat(margins)


def summarize_arm(name, pos_margin, neg_margin, *, base_bundle, adapter_bundle):
    combined = torch.cat([pos_margin, neg_margin])
    adapter_combined = torch.cat([adapter_bundle["pos_margin"], adapter_bundle["neg_margin"]])
    base_combined = torch.cat([base_bundle["pos_margin"], base_bundle["neg_margin"]])
    adapter_gap = gap(adapter_bundle)
    base_gap = gap(base_bundle)
    after_gap = float(pos_margin.mean() - neg_margin.mean())
    pos_one = pos_margin > 0
    neg_one = neg_margin > 0
    adapter_pred = adapter_combined.gt(0)
    base_pred = base_combined.gt(0)
    after_pred = combined.gt(0)
    return {
        "arm": name,
        "gap_after": after_gap,
        "gap_recovery_toward_base": 1.0 - abs(after_gap - base_gap) / max(abs(adapter_gap - base_gap), 1e-8),
        "rule_accuracy": float(torch.cat([pos_one, ~neg_one]).float().mean()),
        "label_changes_vs_adapter": int((after_pred != adapter_pred).sum()),
        "prediction_agreement_with_base": float((after_pred == base_pred).float().mean()),
        "pos_predicts_1": float(pos_one.float().mean()),
        "neg_predicts_1": float(neg_one.float().mean()),
        "both_0": int((~pos_one & ~neg_one).sum()),
        "both_1": int((pos_one & neg_one).sum()),
        "pos1_neg0": int((pos_one & ~neg_one).sum()),
        "pos0_neg1": int((~pos_one & neg_one).sum()),
    }


DIRECTION_ARMS = ("s1", "voice", "shared", "private_s1", "private_voice")
results = []
torch.manual_seed(0)

for task in ("s1", "voice"):
    print(f"Intervening on {task} adapter...")
    tokenizer, model = load_model(str(TASKS[task]["adapter"]), DEVICE)
    eval_pairs = TASKS[task]["eval"]
    adapter_eval = adapter_acts[task]["eval"]
    base_eval = base[task]["eval"]
    local_center_key = f"center_{task}"
    for layer in LAYERS_TO_INTERVENE:
        results.append({
            "task": task,
            "layer": layer,
            **summarize_arm("unablated", adapter_eval["pos_margin"], adapter_eval["neg_margin"],
                            base_bundle=base_eval, adapter_bundle=adapter_eval),
        })
        random_dir = F.normalize(torch.randn(hidden), dim=0)
        extra = {"random": (random_dir, directions[layer][local_center_key])}
        for name in DIRECTION_ARMS:
            extra[name] = (directions[layer][name], directions[layer][local_center_key])
        for name, (direction, center) in extra.items():
            print(f"  L{layer} {task} ← {name}")
            pos_m = evaluate_projection(model, tokenizer, pair_texts(eval_pairs, "pos"), layer=layer, direction=direction, center=center)
            neg_m = evaluate_projection(model, tokenizer, pair_texts(eval_pairs, "neg"), layer=layer, direction=direction, center=center)
            results.append({
                "task": task,
                "layer": layer,
                **summarize_arm(name, pos_m, neg_m, base_bundle=base_eval, adapter_bundle=adapter_eval),
            })
    unload(model, tokenizer)

(OUT / "transfer_results.json").write_text(json.dumps(results, indent=2))
for row in results:
    print(
        f"{row['task']:5s} L{row['layer']} {row['arm']:14s} "
        f"gap={row['gap_after']:7.3f} rec={row['gap_recovery_toward_base']:6.3f} "
        f"rule={row['rule_accuracy']:.3f} flips={row['label_changes_vs_adapter']:3d} "
        f"base_agree={row['prediction_agreement_with_base']:.3f} "
        f"both0/1={row['both_0']}/{row['both_1']}"
    )

In [ ]:
import shutil

fig, axes = plt.subplots(2, 2, figsize=(13, 7.5), constrained_layout=True)
arm_order = ["s1", "voice", "shared", "private_s1", "private_voice", "random"]
for ax_row, task in zip(axes, ("s1", "voice")):
    for ax, layer in zip(ax_row, LAYERS_TO_INTERVENE):
        subset = [row for row in results if row["task"] == task and row["layer"] == layer]
        unablated = next(row for row in subset if row["arm"] == "unablated")
        plotted = [next(row for row in subset if row["arm"] == arm) for arm in arm_order]
        x = np.arange(len(arm_order))
        ax.bar(x, [row["gap_after"] for row in plotted], color="tab:blue")
        ax.axhline(unablated["gap_after"], color="tab:red", linestyle=":", label="Unablated adapter")
        ax.axhline(baseline_eval[f"{task}_base"]["gap"], color="black", linestyle="--", label="Unadapted base")
        ax.set_xticks(x, [arm.replace("_", "\n") for arm in arm_order], fontsize=8)
        ax.set_title(f"{task} model, layer {layer}")
        ax.set_ylabel("pos − neg logit gap")
axes[0, 1].legend(frameon=False, loc="upper right")
fig.suptitle("Fixed-CoT residual projection on held-out val pairs")
fig.savefig(OUT / "transfer_gaps.png", dpi=300, bbox_inches="tight")
plt.show()

run_metadata = {
    "base_model": BASE_MODEL,
    "s1_adapter_sha256": S1_SHA,
    "voice_adapter_sha256": VOICE_SHA,
    "discovery_n": DISCOVERY_N,
    "voice_eval_n": len(voice_eval),
    "s1_eval_n": len(s1_eval),
    "voice_discovery": "data/training_data/synthetic_ethics_voice_paired_train.jsonl",
    "voice_eval": "data/validation_data/synthetic_ethics_voice_paired_val.jsonl",
    "s1_discovery": "data/training_data/synthetic_ethics_cot_training_v2.jsonl",
    "s1_eval": "data/validation_data/synthetic_ethics_cot_val_v2.jsonl",
    "s1_eval_flip_kinds": {k: sum(p["flip"] == k for p in s1_eval) for k in ("lexical", "canonical")},
    "layers_intervened": list(LAYERS_TO_INTERVENE),
    "direction_fit": "train/discovery only; val used only for eval",
    "shared_direction": "first right-singular vector of the 2 x d mean-arrow matrix [d_s1; d_voice]",
    "center": "local adapter discovery mean, not transferred",
}
(OUT / "experiment.json").write_text(json.dumps(run_metadata, indent=2))

archive = shutil.make_archive("/content/residual_s1_voice_transfer_05b", "zip", root_dir=OUT)
files.download(archive)
print("Downloaded", archive)
print("Look in Downloads for residual_s1_voice_transfer_05b.zip")